<a href="https://colab.research.google.com/github/Sahab00/AI-Powered-Health-Monitoring-of-Hive-Royalty/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model Training

In [ ]:
#required library
!apt-get install -y unrar
!unrar x "/content/drive/MyDrive/FypDataset/all_from_zero_my_dataset1.rar" "/content/"


In [ ]:
import os
import shutil
import random

dataset_path = "/content/New folder"  # my orignal folder wiht images and lables
images_path = os.path.join(dataset_path, "images")
labels_path = os.path.join(dataset_path, "labels")


train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1


# CREATE FOLDERS

split_folders = ["train", "val", "test"]
for split in split_folders:
    os.makedirs(os.path.join(images_path, split), exist_ok=True)
    os.makedirs(os.path.join(labels_path, split), exist_ok=True)

# GET FILES

image_files = [f for f in os.listdir(images_path) if f.endswith((".jpg", ".png", ".jpeg"))]
random.shuffle(image_files)

num_images = len(image_files)
train_end = int(train_ratio * num_images)
val_end = train_end + int(val_ratio * num_images)

train_files = image_files[:train_end]
val_files = image_files[train_end:val_end]
test_files = image_files[val_end:]


# MOVE FILES

def move_files(file_list, split):
    for img_file in file_list:
        label_file = os.path.splitext(img_file)[0] + ".txt"  # YOLO format label
        # move image
        shutil.move(os.path.join(images_path, img_file),
                    os.path.join(images_path, split, img_file))
        # move label
        if os.path.exists(os.path.join(labels_path, label_file)):
            shutil.move(os.path.join(labels_path, label_file),
                        os.path.join(labels_path, split, label_file))
        else:
            print(f"⚠️ Label missing for {img_file}")

move_files(train_files, "train")
move_files(val_files, "val")
move_files(test_files, "test")

print("Dataset split completed!")
print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")



In [ ]:
#required library
!pip install -U ultralytics

In [ ]:

from ultralytics import YOLO
import torch
import os

model = YOLO("yolo11s-seg.pt")
print("YOLOv11 Segmentation model loaded")

model.train(
    data="/content/New folder/data.yaml",
    epochs=120,
    imgsz=640,


    batch=256,        # Use full GPU VRAM if memory allows
    workers=16,       # Keep CPU feeding GPU efficiently
    device=0,
    amp=True,         # Mixed precision = faster
    cache=True,       # Loads all images into RAM

    # Training mode
    optimizer="AdamW",
    lr0=0.0015,       # Slightly higher LR can converge faster
    patience=15,      # Early stopping sooner
    pretrained=True,
    project="bee_mite_segmentation",
    name="yolov11_seg_bee_a100_fast",
    verbose=True
)

In [ ]:
last = df.iloc[-1]

print("Precision:", last["metrics/precision(B)"])
print("Recall:", last["metrics/recall(B)"])
print("mAP@50:", last["metrics/mAP50(B)"])
print("mAP@50-95:", last["metrics/mAP50-95(B)"])
from ultralytics import YOLO

# Load best trained weights
model = YOLO(
    "bee_mite_segmentation/yolov11_seg_bee_a100_fast/weights/best.pt"
)
metrics = model.val()

print("Precision:", metrics.box.map)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

